# MediSense — 14개 의료 데이터셋 → 공통 SFT Dataset 변환 파이프라인

---

# CELL 1. 프로젝트 설명 및 전체 파이프라인

**이 노트북은 모델 학습을 하지 않는다.** HF 데이터셋 11개 + AI Hub 로컬 데이터셋 3개,
총 14개를 하나의 공통 포맷으로 변환/정제/병합해서 `train.jsonl`/`validation.jsonl`/`test.jsonl`
(+ Hugging Face `Dataset` 디렉터리)로 저장하는 것까지가 범위다. 학습은 이 결과물을 불러와서
별도 노트북(`main_train_llm_lora_v2.ipynb` 계열)에서 진행한다.

AI Hub 데이터셋은 원본 zip/tar 압축 그대로 Drive에 올려둔 상태라, HF 데이터셋을 만지기 전에
**먼저 압축부터 풀어야** 뒤에서 폴더 구조를 확인할 수 있다 — 그래서 압축 해제(CELL 6)를
CONFIG 바로 다음, HF 로그인보다 앞에 둔다.

## 아키텍처 / 데이터 흐름

```
[Drive: 압축파일 (zip/tar/tar.gz/tgz/gz/7z)]
              │
     CELL 6. 압축 탐색 + 자동 해제
     (EXTRACT_ROOT에 풀고, AIHUB_DATASETS.path 자동 연결)
              │
              ▼
[HF Datasets 11개]                    [AI Hub 압축 해제 결과]
        │                                          │
   load_dataset() (CELL 8)              폴더/파일 재탐색 (CELL 10)
        │                                          │
   구조/컬럼/샘플 확인 (CELL 9)              JSON/JSONL/CSV 구조·샘플 확인 (CELL 11)
        │                                          │
        ▼                                          ▼
  QAAdapter / MCQAdapter / DialogueAdapter    AIHubAdapter
  (CELL 14/15/16) + ReasoningAdapter(17)      (CELL 18, CONFIG 매핑 기반)
        │                                          │
        └────────────────┬─────────────────────────┘
                          ▼
              공통 messages 포맷으로 변환
        (+ source_dataset / data_type / format_type)
                (CELL 19, CELL 20)
                          │
                          ▼
              데이터 품질 검증 + 통계 (CELL 21~22)
                          │
                          ▼
              14개 데이터셋 병합 (CELL 23)
                          │
                          ▼
       Cross-Dataset 중복 제거 — 우선순위 기반 (CELL 24)
                          │
                          ▼
       data_type 비율 확인 + 선택적 Mixing (CELL 25)
                          │
                          ▼
   Train(80) / Validation(10) / Test(10) Stratified Split (CELL 26)
                          │
                          ▼
         최종 검증 (CELL 27) → JSONL + HF Dataset 저장 (CELL 28~29)
```

## 설계 원칙

- **컬럼명을 임의로 가정하지 않는다** — HF 데이터셋은 CELL 9에서, AI Hub 데이터셋은
  CELL 11에서 실제 구조를 먼저 출력하고, 그 결과를 보고 CONFIG를 맞추는 순서로 간다.
- **하나가 실패해도 전체가 죽지 않는다** — 압축파일 단위/데이터셋 단위로 try/except,
  실패하면 `FAILED_ARCHIVES`/`FAILED_DATASETS`에 사유를 남기고 나머지는 계속 진행한다.
- **로그는 `[INFO]`/`[WARNING]`/`[ERROR]`** 접두사로 통일한다.
- **오버샘플링으로 데이터를 복제하지 않는다** — Mixing은 다운샘플링만 한다.
- **큰 JSON 파일을 통째로 메모리에 올리지 않는다** — 50MB 이상이면 `ijson`으로 스트리밍
  파싱한다(CELL 11, CELL 20).
- **원천데이터/라벨링데이터가 나뉜 AI Hub 데이터셋 대응** — 데이터셋 하나에 압축파일이
  여러 개 연결될 수 있으므로 `AIHUB_DATASETS[].archive_names`는 리스트다.

---

# CELL 2. Google Drive Mount

압축파일 원본(`ARCHIVE_ROOT`), 압축 해제 결과(`EXTRACT_ROOT`), 최종 산출물(`OUTPUT_DIR`)이
전부 Drive에 있으므로 가장 먼저 마운트한다. 압축 해제 결과도 Drive에 남기므로, Colab
런타임이 끊겨도 다시 압축을 풀 필요가 없다(CELL 6에서 이미 풀린 폴더는 건너뜀).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("[INFO] Google Drive 마운트 완료")


---

# CELL 3. 라이브러리 설치

이 노트북은 학습을 하지 않으므로 `transformers`/`peft`/`trl`/`bitsandbytes`는 필요 없다.
`ijson`은 대용량 JSON 스트리밍 파싱(CELL 11, 20)에, `py7zr`은 `.7z` 압축 해제(CELL 6)에
쓴다 — 둘 다 없어도 나머지 기능은 동작하고, 이 두 기능만 필요할 때 경고와 함께 우회한다.

**`pandas`는 설치 목록에 넣지 않는다** — Colab에 이미 깔려있고, `google-colab`/`dask-cudf-cu12`/
`cudf-cu12` 같은 Colab 내장 패키지가 특정 pandas 버전에 맞춰져 있어서, 여기서 최신으로
업그레이드하면 오히려 그 패키지들과 충돌한다(과거 numpy 강제 다운그레이드 때와 같은 종류의
문제). 이 노트북은 `read_csv`/`read_parquet`/`to_dict` 같은 기본 기능만 쓰므로 Colab
기본 버전 그대로 써도 충분하다.

## 설치 후 "런타임 → 세션 다시 시작"을 한 번 하세요.

In [ ]:
%pip install -q -U datasets huggingface_hub ijson py7zr
print("[INFO] 설치 완료 — 런타임을 다시 시작한 뒤 CELL 4부터 이어서 실행하세요.")


---

# CELL 4. Import 및 환경 설정

In [ ]:
import os
import re
import json
import glob
import gzip
import shutil
import random
import tarfile
import zipfile
import unicodedata
from collections import Counter, defaultdict

import pandas as pd
from datasets import Dataset, load_dataset, concatenate_datasets

print("[INFO] Import 완료")


---

# CELL 5. CONFIG 설정

이 노트북에서 바꿀 일이 있는 값은 전부 여기 모아둔다.

- `HF_DATASETS`/`HF_COLUMN_OVERRIDE`: HF Hub 데이터셋 11개와 컬럼 매핑(이전 세션에서 이미
  검증된 값이라 미리 채워뒀다).
- `ARCHIVE_ROOT`/`EXTRACT_ROOT`/`AIHUB_DATASETS`/`AIHUB_COLUMN_MAPPING`: AI Hub 로컬(Drive)
  데이터셋 3개. `archive_names`는 **리스트**다 — AI Hub 데이터셋은 원천데이터/라벨링데이터가
  별도 압축파일로 나뉜 경우가 있어서, 데이터셋 하나에 압축파일이 여러 개 연결될 수 있다.
  실제 파일명은 아직 모르므로 CELL 6 실행 결과(`[INFO] Found archive: ...`)를 보고 채운다.

In [ ]:
SEED = 42

# ------------------------------------------------------------
# 5-1. Hugging Face 데이터셋 (11개)
# ------------------------------------------------------------
HF_DATASETS = [
    {"name": "ChuGyouk/Asan-AMC-Healthinfo", "data_type": "medical_knowledge", "format_type": "qa_pair", "max_samples": 2000, "hf_config": None},
    {"name": "ChuGyouk/MedQA", "data_type": "medical_knowledge", "format_type": "mcq_abcd", "max_samples": 2000, "hf_config": "ko"},
    {"name": "ChuGyouk/KoMedInstruct-52k", "data_type": "medical_knowledge", "format_type": "qa_pair", "max_samples": 2000, "hf_config": None},
    {"name": "snuh/ClinicalQA", "data_type": "medical_knowledge", "format_type": "mcq_lettered", "max_samples": None, "hf_config": None},
    {"name": "ChuGyouk/AI_healthcare_QA", "data_type": "medical_knowledge", "format_type": "qa_pair", "max_samples": 2000, "hf_config": None},
    {"name": "ChuGyouk/HealthSearchQA-ko", "data_type": "medical_knowledge", "format_type": "qa_pair", "max_samples": None, "hf_config": None},
    {"name": "hcw0329/medical-korean-alpaca", "data_type": "medical_knowledge", "format_type": "qa_pair", "max_samples": 2000, "hf_config": None},
    {"name": "ChuGyouk/medical-o1-reasoning-SFT-Ko", "data_type": "medical_reasoning", "format_type": "qa_pair", "max_samples": 2000, "hf_config": None},
    {"name": "ChuGyouk/ChainOfDiagnosis-Ko", "data_type": "medical_reasoning", "format_type": "dialogue", "max_samples": 2000, "hf_config": None},
    {"name": "ChuGyouk/MedQA-Evol-Korean", "data_type": "medical_reasoning", "format_type": "qa_pair", "max_samples": 2000, "hf_config": None},
    {"name": "squarelike/ko_medical_chat", "data_type": "conversation_style", "format_type": "dialogue", "max_samples": None, "hf_config": None},
]

# qa_pair 데이터셋의 질문/답변 컬럼 — 자동판별(CELL 9)이 실패하거나 틀리면 여기를 고친다.
# snuh/ClinicalQA는 mcq_lettered 전용 키(options_col/answer_col/explanation_col)를 쓴다 —
# CELL 9 확인 결과 answer가 'C'처럼 글자 하나뿐이라 options 딕셔너리와 조합해야 함(CELL 15).
HF_COLUMN_OVERRIDE = {
    "ChuGyouk/Asan-AMC-Healthinfo": {"question_col": "instruction", "input_col": None, "answer_col": "output"},
    "ChuGyouk/KoMedInstruct-52k": {"question_col": "instruction", "input_col": "input", "answer_col": "output"},
    "snuh/ClinicalQA": {"question_col": "question", "options_col": "options", "answer_col": "answer", "explanation_col": "explanation"},
    "ChuGyouk/AI_healthcare_QA": {"question_col": "question", "input_col": None, "answer_col": "gpt4o"},
    "ChuGyouk/HealthSearchQA-ko": {"question_col": "question_ko", "input_col": None, "answer_col": "answer_ko"},
    "hcw0329/medical-korean-alpaca": {"question_col": "instruction", "input_col": "input", "answer_col": "output"},
    "ChuGyouk/medical-o1-reasoning-SFT-Ko": {"question_col": "Question", "input_col": None, "answer_col": "Response"},
    "ChuGyouk/MedQA-Evol-Korean": {"question_col": "input", "input_col": None, "answer_col": "output"},
}

MCQ_OPTION_KEYS = ["A", "B", "C", "D"]
MCQ_ANSWER_IDX_COL = "answer_idx"
MCQ_ANSWER_TEXT_COL = "answer"

REASONING_SOURCE_DATASETS = {"ChuGyouk/medical-o1-reasoning-SFT-Ko", "ChuGyouk/ChainOfDiagnosis-Ko"}

# ------------------------------------------------------------
# 5-2. AI Hub 압축파일 / 압축 해제 경로
# ------------------------------------------------------------
ARCHIVE_ROOT = "/content/drive/MyDrive/dataset"
EXTRACT_ROOT = "/content/drive/MyDrive/MediSense/extracted"
FORCE_REEXTRACT = False

# archive_names: 이 데이터셋에 해당하는 압축파일 이름(확장자 포함, 여러 개 가능 — 원천/라벨링
# 데이터가 나뉜 경우). path는 CELL 6이 압축 해제 후 자동으로 채운다(직접 안 건드려도 됨).
# 실제 파일명 3개 확인 완료(사용자 Drive 탐색 결과) — 전부 단일 .7z(분할 아님).
AIHUB_DATASETS = [
    {"name": "aihub_professional_medical_knowledge", "archive_names": ["08.전문 의학지식 데이터.7z"],
     "path": None, "data_type": "medical_knowledge", "format_type": "aihub"},
    {"name": "aihub_essential_medical_knowledge", "archive_names": ["09.필수의료 의학지식 데이터.7z"],
     "path": None, "data_type": "medical_knowledge", "format_type": "aihub"},
    {"name": "aihub_healthcare_qa", "archive_names": ["120.초거대AI 사전학습용 헬스케어 질의응답 데이터.7z"],
     "path": None, "data_type": "medical_knowledge", "format_type": "aihub"},
]

# root_key: JSON 최상위에서 실제 레코드 리스트까지 가는 경로(점(.)으로 구분, 최상위가 이미
# 리스트면 None). question_key/answer_key 또는 dialogue_key 중 해당하는 쪽만 채우면 된다.
# 전부 CELL 11 결과를 보고 채우는 자리 — 지금은 placeholder.
AIHUB_COLUMN_MAPPING = {
    "aihub_professional_medical_knowledge": {"root_key": None, "question_key": None, "answer_key": None, "dialogue_key": None},
    "aihub_essential_medical_knowledge": {"root_key": None, "question_key": None, "answer_key": None, "dialogue_key": None},
    "aihub_healthcare_qa": {"root_key": None, "question_key": None, "answer_key": None, "dialogue_key": None},
}

# 이 크기(MB) 이상인 JSON 파일은 통째로 읽지 않고 ijson으로 스트리밍 파싱한다.
LARGE_JSON_THRESHOLD_MB = 50

# ------------------------------------------------------------
# 5-3. 전처리 옵션
# ------------------------------------------------------------
MIN_USER_LENGTH = 5
MIN_ASSISTANT_LENGTH = 5
MAX_USER_LENGTH = 2000
MAX_ASSISTANT_LENGTH = 4000

ENABLE_REASONING_STRIP = True
ENABLE_DUPLICATE_REMOVAL = True
ENABLE_CROSS_DATASET_DUPLICATE_REMOVAL = True
ENABLE_PII_MASKING = True

# ------------------------------------------------------------
# 5-4. Cross-Dataset 중복 제거 우선순위 (숫자가 작을수록 우선)
# ------------------------------------------------------------
DATASET_PRIORITY = [
    "aihub_professional_medical_knowledge", "aihub_essential_medical_knowledge", "aihub_healthcare_qa",
    "snuh/ClinicalQA",
    "ChuGyouk/Asan-AMC-Healthinfo",
    "ChuGyouk/KoMedInstruct-52k",
    "ChuGyouk/AI_healthcare_QA", "ChuGyouk/HealthSearchQA-ko", "hcw0329/medical-korean-alpaca", "ChuGyouk/MedQA",
    "ChuGyouk/medical-o1-reasoning-SFT-Ko", "ChuGyouk/ChainOfDiagnosis-Ko", "ChuGyouk/MedQA-Evol-Korean",
    "squarelike/ko_medical_chat",
]

# ------------------------------------------------------------
# 5-5. Mixing (초기엔 끔 — 원본 비율부터 확인)
# ------------------------------------------------------------
ENABLE_MIXING = False
TARGET_RATIOS = {"medical_knowledge": 0.60, "medical_reasoning": 0.25, "conversation_style": 0.15}

# ------------------------------------------------------------
# 5-6. 출력 경로
# ------------------------------------------------------------
OUTPUT_DIR = "/content/drive/MyDrive/MediSense/processed_dataset"

print(f"[INFO] HF_DATASETS: {len(HF_DATASETS)}개, AIHUB_DATASETS: {len(AIHUB_DATASETS)}개 (총 {len(HF_DATASETS)+len(AIHUB_DATASETS)}개)")
print(f"[INFO] SEED={SEED}, ARCHIVE_ROOT={ARCHIVE_ROOT}, EXTRACT_ROOT={EXTRACT_ROOT}, OUTPUT_DIR={OUTPUT_DIR}")


---

# CELL 6. AI Hub 압축파일 탐색 및 자동 압축 해제

`ARCHIVE_ROOT`(원본 압축파일이 있는 곳)를 재귀적으로 뒤져서 `.zip`/`.tar`/`.tar.gz`/`.tgz`/
`.gz`/`.7z`(가능한 경우)를 전부 찾고, 압축파일 이름 기준으로 `EXTRACT_ROOT` 밑 별도 폴더에
해제한다. 이미 해제된 폴더가 있으면 `FORCE_REEXTRACT=True`가 아닌 한 다시 풀지 않는다.
압축 해제 결과(Drive)는 Colab 런타임이 끊겨도 남는다.

**중첩 압축까지 처리한다** — AI-Hub는 원천/라벨링 데이터를 국문/영문·진료과별 zip으로 한 번
더 묶어 배포하는 경우가 있다(`01.원천데이터/TS_국문_기타.zip`처럼). 최상위 압축을 푼 뒤에도
`EXTRACT_ROOT` 안에 남아있는 압축파일을 새로 발견되는 게 없을 때까지 반복 탐색해서 전부
풀어낸다.

해제가 끝나면 `AIHUB_DATASETS[].archive_names`와 이름이 일치하는 압축 해제 폴더를 찾아
`path`에 자동으로 연결한다 — 원천데이터/라벨링데이터처럼 데이터셋 하나에 압축파일이 여러 개
연결되면 `path`가 리스트가 된다. 실패한 압축파일은 `FAILED_ARCHIVES`에 남기고 계속 진행한다.

In [ ]:
def _fix_mojibake_name(name):
    """압축 내부 파일명이 cp437로 잘못 디코딩돼 한글이 깨진 경우 복구를 시도한다."""
    try:
        return name.encode("cp437").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return name


_SPLIT_PART_RE = re.compile(r"\.(7z|zip)\.(\d{3})$", re.IGNORECASE)


def _archive_stem_and_format(path):
    """확장자를 뗀 stem과 형식을 반환한다. 한글 파일명은 NFC로 정규화해서 돌려준다 —
    Google Drive가 넘겨주는 파일명이 NFD(자모 분리형)일 수 있어서, CONFIG에 직접 타이핑한
    NFC 문자열과 바이트 단위로 다르게 보여 매칭이 실패하는 걸 막기 위함."""
    lower = path.lower()
    split_match = _SPLIT_PART_RE.search(lower)
    if split_match:
        # "xxx.7z.001" -> stem="xxx", fmt="7z" (뒤 조각들은 py7zr/7z가 같은 폴더에서 알아서 찾음)
        fmt = split_match.group(1)
        stem = path[: -len(split_match.group(0))]
    else:
        stem = None
        fmt = None
        for ext, candidate_fmt in ((".tar.gz", "tar.gz"), (".tgz", "tar.gz"), (".tar", "tar"), (".7z", "7z"), (".zip", "zip"), (".gz", "gz")):
            if lower.endswith(ext):
                stem, fmt = path[: -len(ext)], candidate_fmt
                break
        if stem is None:
            return None, None
    return unicodedata.normalize("NFC", stem), fmt


def find_archives(root):
    """일반 압축파일 + 분할 압축(.7z.001/.zip.001 등)을 찾는다.
    분할 압축은 첫 조각(001)만 등록한다 — 나머지 조각(002, 003...)은 압축 해제 시
    같은 폴더에서 라이브러리가 자동으로 찾아 읽는다."""
    exts = (".zip", ".tar", ".tar.gz", ".tgz", ".gz", ".7z")
    found = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            lower = fn.lower()
            split_match = _SPLIT_PART_RE.search(lower)
            if split_match:
                if split_match.group(2) == "001":
                    found.append(os.path.join(dirpath, fn))
                continue  # 002, 003... 은 별도 아카이브로 취급하지 않음
            if lower.endswith(exts):
                found.append(os.path.join(dirpath, fn))
    return found


def extract_archive(path, dest_dir, fmt):
    os.makedirs(dest_dir, exist_ok=True)
    if fmt == "zip":
        with zipfile.ZipFile(path, "r") as zf:
            for info in zf.infolist():
                fixed_name = _fix_mojibake_name(info.filename)
                target_path = os.path.join(dest_dir, fixed_name)
                if info.is_dir():
                    os.makedirs(target_path, exist_ok=True)
                    continue
                os.makedirs(os.path.dirname(target_path), exist_ok=True)
                with zf.open(info) as src, open(target_path, "wb") as dst:
                    dst.write(src.read())
    elif fmt in ("tar", "tar.gz"):
        mode = "r:gz" if fmt == "tar.gz" else "r"
        with tarfile.open(path, mode) as tf:
            tf.extractall(dest_dir)
    elif fmt == "gz":
        # 디렉터리 아카이브가 아니라 파일 하나짜리 gzip인 경우
        out_name = os.path.basename(path)[:-3] or "extracted.bin"
        with gzip.open(path, "rb") as src, open(os.path.join(dest_dir, out_name), "wb") as dst:
            shutil.copyfileobj(src, dst)
    elif fmt == "7z":
        import py7zr  # CELL 3에서 설치 안 됐으면 여기서 ImportError -> 아래 except에서 FAILED_ARCHIVES로 기록됨
        with py7zr.SevenZipFile(path, mode="r") as z:
            z.extractall(dest_dir)
    else:
        raise ValueError(f"지원하지 않는 압축 형식: {path}")


def _dir_stats(path):
    file_count = 0
    total_size = 0
    for dirpath, _, filenames in os.walk(path):
        for fn in filenames:
            file_count += 1
            total_size += os.path.getsize(os.path.join(dirpath, fn))
    return file_count, total_size


FAILED_ARCHIVES = []
ARCHIVE_EXTRACT_LOG = []

if not os.path.isdir(ARCHIVE_ROOT):
    print(f"[ERROR] ARCHIVE_ROOT 경로가 없습니다: {ARCHIVE_ROOT} — CELL 5에서 실제 Drive 경로로 수정하세요.")
else:
    archive_paths = find_archives(ARCHIVE_ROOT)
    print(f"[INFO] {ARCHIVE_ROOT} 밑에서 압축파일 {len(archive_paths)}개 발견")
    for ap in archive_paths:
        split_tag = " (분할 압축 첫 조각 — 나머지는 같은 폴더에서 자동으로 읽음)" if _SPLIT_PART_RE.search(ap.lower()) else ""
        print(f"[INFO] Found archive: {os.path.relpath(ap, ARCHIVE_ROOT)}{split_tag}")

    os.makedirs(EXTRACT_ROOT, exist_ok=True)

    for ap in archive_paths:
        stem, fmt = _archive_stem_and_format(ap)
        if stem is None:
            continue
        dest_dir = os.path.join(EXTRACT_ROOT, os.path.basename(stem))

        already_done = os.path.isdir(dest_dir) and os.listdir(dest_dir)
        if already_done and not FORCE_REEXTRACT:
            file_count, total_size = _dir_stats(dest_dir)
            print(f"[INFO] 이미 압축 해제됨, 건너뜀: {dest_dir} (파일 {file_count}개, {total_size/1024/1024:.1f}MB)")
            ARCHIVE_EXTRACT_LOG.append({"archive": ap, "format": fmt, "dest": dest_dir, "status": "skipped_existing",
                                          "file_count": file_count, "total_size_mb": round(total_size / 1024 / 1024, 1)})
            continue

        if already_done and FORCE_REEXTRACT:
            shutil.rmtree(dest_dir)

        try:
            extract_archive(ap, dest_dir, fmt)
            file_count, total_size = _dir_stats(dest_dir)
            print(f"[INFO] 압축 해제 성공: {os.path.basename(ap)} ({fmt}) -> {dest_dir}")
            print(f"        파일 {file_count}개, 총 {total_size/1024/1024:.1f}MB")
            ARCHIVE_EXTRACT_LOG.append({"archive": ap, "format": fmt, "dest": dest_dir, "status": "success",
                                          "file_count": file_count, "total_size_mb": round(total_size / 1024 / 1024, 1)})
        except Exception as e:
            print(f"[ERROR] 압축 해제 실패: {ap}: {type(e).__name__}: {e}")
            FAILED_ARCHIVES.append({"archive": ap, "reason": str(e)})
            ARCHIVE_EXTRACT_LOG.append({"archive": ap, "format": fmt, "dest": dest_dir, "status": "failed", "reason": str(e)})

    # 압축을 풀고 나서도 그 안에 또 압축파일이 있는 경우(AI-Hub가 원천/라벨링 데이터를
    # 국문/영문·진료과별 zip으로 한 번 더 묶어 배포하는 방식) 대응 — 새로 발견되는 압축파일이
    # 없을 때까지 반복해서 몇 겹으로 중첩돼 있어도 전부 풀어낸다. 같은 폴더 안에 압축파일명
    # 하위 폴더로 해제해서(형제 위치) 기존 카테고리별 디렉터리 구조를 그대로 유지한다.
    for _pass in range(5):
        nested = find_archives(EXTRACT_ROOT)
        newly_extracted = 0
        for ap in nested:
            stem, fmt = _archive_stem_and_format(ap)
            if stem is None:
                continue
            dest_dir = stem
            if os.path.isdir(dest_dir) and os.listdir(dest_dir):
                continue
            try:
                extract_archive(ap, dest_dir, fmt)
                newly_extracted += 1
                file_count, total_size = _dir_stats(dest_dir)
                print(f"[INFO] (중첩) 압축 해제 성공: {os.path.relpath(ap, EXTRACT_ROOT)} "
                      f"(파일 {file_count}개, {total_size/1024/1024:.1f}MB)")
            except Exception as e:
                print(f"[ERROR] (중첩) 압축 해제 실패: {ap}: {type(e).__name__}: {e}")
                FAILED_ARCHIVES.append({"archive": ap, "reason": str(e)})
        if newly_extracted == 0:
            break

    # archive_names <-> 압축 해제 폴더 자동 연결. 이름이 여러 개(원천/라벨링) 매칭되면 path가 리스트가 된다.
    for d in AIHUB_DATASETS:
        matched = []
        for name_hint in d["archive_names"]:
            if not name_hint or str(name_hint).startswith("<"):
                continue
            hint_stem, _ = _archive_stem_and_format(name_hint)
            hint_stem = hint_stem or name_hint
            for entry in ARCHIVE_EXTRACT_LOG:
                if entry["status"] not in ("success", "skipped_existing"):
                    continue
                entry_stem, _ = _archive_stem_and_format(os.path.basename(entry["archive"]))
                if entry_stem == hint_stem:
                    matched.append(entry["dest"])
        if matched:
            d["path"] = matched if len(matched) > 1 else matched[0]
            print(f"[INFO] {d['name']}: path 자동 연결 -> {d['path']}")
        else:
            print(f"[WARNING] {d['name']}: archive_names={d['archive_names']}에 매칭되는 압축 해제 폴더가 "
                  f"없습니다 — 위 'Found archive' 목록을 보고 CELL 5의 archive_names를 실제 파일명으로 고치세요.")

    if FAILED_ARCHIVES:
        print(f"\n[WARNING] 압축 해제 실패 {len(FAILED_ARCHIVES)}건: {FAILED_ARCHIVES}")


---

# CELL 7. Hugging Face 로그인

일부 데이터셋(gated repo)이 있을 수 있으니 로그인해둔다. Colab Secrets에 `HF_TOKEN`을
등록해야 한다(왼쪽 메뉴 열쇠 아이콘 → Secrets).

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    print("[WARNING] HF_TOKEN이 없습니다 — 비공개/gated 데이터셋이 있으면 CELL 8에서 실패할 수 있습니다.")
else:
    login(token=HF_TOKEN)
    print("[INFO] Hugging Face 로그인 완료")


---

# CELL 8. HF 데이터셋 로드

`HF_DATASETS`에 등록된 순서대로 로드한다. `max_samples`가 있으면 로드 직후 SEED 기준
랜덤 샘플링(앞에서부터 slice하지 않음)으로 상한을 맞춘다. 하나가 실패해도 나머지는 계속
진행하고, 실패 사유는 `FAILED_DATASETS`에 남긴다.

In [ ]:
raw_hf_datasets = {}
FAILED_DATASETS = []

for d in HF_DATASETS:
    name = d["name"]
    try:
        # 일부 데이터셋(MedQA 등)은 language/version별 config 이름을 명시해야 로드된다.
        if d.get("hf_config"):
            ds = load_dataset(name, d["hf_config"], split="train", token=HF_TOKEN if HF_TOKEN else None)
        else:
            ds = load_dataset(name, split="train", token=HF_TOKEN if HF_TOKEN else None)
        original_n = len(ds)
        if d["max_samples"]:
            ds = ds.shuffle(seed=SEED).select(range(min(d["max_samples"], original_n)))
        raw_hf_datasets[name] = ds
        cap_note = f"(원본 {original_n}개 중 랜덤 {len(ds)}개 샘플링)" if d["max_samples"] and original_n > d["max_samples"] else "(전량)"
        print(f"[INFO] {name}: {len(ds)}개 로드 {cap_note}")
    except Exception as e:
        print(f"[ERROR] {name} 로드 실패: {type(e).__name__}: {e}")
        FAILED_DATASETS.append({"name": name, "stage": "load", "reason": str(e)})

print(f"\n[INFO] HF 데이터셋 로드 성공: {len(raw_hf_datasets)}/{len(HF_DATASETS)}개")


---

# CELL 9. HF 데이터셋 구조 분석 및 샘플 확인

각 데이터셋의 컬럼 목록과 첫 샘플을 실제로 출력한다. `format_type="qa_pair"`인 경우
`HF_COLUMN_OVERRIDE`에 지정한 컬럼이 실제로 존재하는지 여기서 확인하고, 틀렸으면 CELL 5로
돌아가 고친다.

In [ ]:
for d in HF_DATASETS:
    name = d["name"]
    if name not in raw_hf_datasets:
        continue
    ds = raw_hf_datasets[name]
    print("=" * 90)
    print(f"{name}  (data_type={d['data_type']}, format_type={d['format_type']})")
    print(f"  개수: {len(ds)}")
    print(f"  컬럼: {ds.column_names}")

    if d["format_type"] == "qa_pair":
        override = HF_COLUMN_OVERRIDE.get(name)
        if override is None:
            print("  [WARNING] HF_COLUMN_OVERRIDE에 이 데이터셋 항목이 없습니다 — CELL 14(QAAdapter)의 "
                  "자동판별에만 의존하게 됩니다.")
        else:
            for key, label in (("question_col", "question_col"), ("answer_col", "answer_col")):
                col = override.get(key)
                status = "OK" if col in ds.column_names else "!! 컬럼 없음 — HF_COLUMN_OVERRIDE 수정 필요"
                print(f"  {label}='{col}' -> {status}")
    elif d["format_type"] == "mcq_abcd":
        missing = [k for k in MCQ_OPTION_KEYS if k not in ds.column_names]
        print(f"  선택지 컬럼 A~D 존재 여부: {'OK' if not missing else f'!! 없음: {missing}'}")
    elif d["format_type"] == "mcq_lettered":
        override = HF_COLUMN_OVERRIDE.get(name, {})
        for key, label in (("question_col", "question_col"), ("options_col", "options_col"), ("answer_col", "answer_col")):
            col = override.get(key)
            status = "OK" if col in ds.column_names else "!! 컬럼 없음 — HF_COLUMN_OVERRIDE 수정 필요"
            print(f"  {label}='{col}' -> {status}")

    print(f"  샘플[0]: {ds[0]}")


---

# CELL 10. AI Hub 압축 해제 결과 및 파일 탐색

CELL 6에서 압축을 푼 `EXTRACT_ROOT` 전체를 재귀적으로 뒤져서 `.json`/`.jsonl`/`.csv`/
`.parquet` 파일을 찾는다. 폴더별 파일 개수와 함께, 파일별 경로/크기/확장자를 출력한다 — 이
결과를 보고 CELL 5의 `AIHUB_COLUMN_MAPPING`을 채운다.

In [ ]:
if not os.path.isdir(EXTRACT_ROOT):
    print(f"[ERROR] EXTRACT_ROOT가 없습니다: {EXTRACT_ROOT} — CELL 6이 성공했는지 확인하세요.")
    by_folder = {}
else:
    data_files = []
    for ext in ("*.json", "*.jsonl", "*.csv", "*.parquet"):
        data_files.extend(glob.glob(os.path.join(EXTRACT_ROOT, "**", ext), recursive=True))

    def _group_key(fp):
        """데이터셋명 + 원천/라벨링 구분으로 묶는다 — 최상위 폴더 하나로만 묶으면 원천/라벨링
        데이터가 섞여서 먼저 찾은 쪽(대개 원천데이터)만 CELL 11에서 샘플로 보이는 문제가 있어,
        경로에 '원천데이터'/'라벨링데이터'가 보이면 그걸로 한 번 더 나눈다."""
        rel = os.path.relpath(fp, EXTRACT_ROOT)
        top = rel.split(os.sep)[0]
        if "원천데이터" in rel:
            return f"{top} / 원천데이터"
        if "라벨링데이터" in rel:
            return f"{top} / 라벨링데이터"
        return top

    by_folder = defaultdict(list)
    for fp in data_files:
        by_folder[_group_key(fp)].append(fp)

    print(f"[INFO] EXTRACT_ROOT 하위에서 데이터 파일 {len(data_files)}개 발견 (폴더 {len(by_folder)}개)\n")
    for folder, files in by_folder.items():
        total_size = sum(os.path.getsize(fp) for fp in files)
        print(f"[폴더] {folder}: 파일 {len(files)}개, 총 {total_size/1024/1024:.1f}MB")
        for fp in files[:5]:
            size_mb = os.path.getsize(fp) / 1024 / 1024
            ext = os.path.splitext(fp)[1]
            print(f"    {fp}  ({ext}, {size_mb:.2f}MB)")
        if len(files) > 5:
            print(f"    ... 외 {len(files) - 5}개")

    if not data_files:
        print("[WARNING] json/jsonl/csv/parquet 파일을 하나도 못 찾았습니다 — 압축 내부 구조가 예상과 "
              "다르거나 폴더가 더 깊이 중첩돼 있을 수 있습니다. EXTRACT_ROOT를 직접 열어보세요.")


---

# CELL 11. AI Hub JSON 구조 분석 및 샘플 확인

CELL 10에서 찾은 폴더별로 대표 파일 하나씩을 열어서 최상위 타입(list/dict), dict라면 key
목록, 실제 레코드로 보이는 샘플 3~5개를 출력한다. **`LARGE_JSON_THRESHOLD_MB`(기본 50MB)
이상인 JSON 파일은 통째로 읽지 않고 `ijson`으로 스트리밍 파싱**해서 앞부분 샘플만 본다
(`ijson`이 없으면 경고 후 전체 로드로 대체). 이 결과를 보고 CELL 5의
`AIHUB_DATASETS[].path`와 `AIHUB_COLUMN_MAPPING`을 채운다.

In [ ]:
def peek_json_structure(file_path, root_key=None, max_samples=3):
    print(f"--- {file_path} ---")
    size_mb = os.path.getsize(file_path) / 1024 / 1024
    print(f"  크기: {size_mb:.1f}MB")
    try:
        if file_path.endswith(".jsonl"):
            records = []
            with open(file_path, encoding="utf-8-sig") as f:
                for i, line in enumerate(f):
                    if i >= max_samples:
                        break
                    line = line.strip()
                    if line:
                        records.append(json.loads(line))
            print(f"  형식: JSONL (앞부분만 읽음)")
            for rec in records:
                print("  샘플:", rec)

        elif file_path.endswith(".csv"):
            df = pd.read_csv(file_path, nrows=max_samples + 5)
            print(f"  형식: CSV, 컬럼: {list(df.columns)}")
            print(df.head(max_samples))

        elif file_path.endswith(".parquet"):
            df = pd.read_parquet(file_path)
            print(f"  형식: Parquet, 컬럼: {list(df.columns)}, 전체 {len(df)}행")
            print(df.head(max_samples))

        elif size_mb >= LARGE_JSON_THRESHOLD_MB:
            try:
                import ijson
                prefix = f"{root_key}.item" if root_key else "item"
                print(f"  형식: JSON (대용량, ijson 스트리밍 파싱, prefix='{prefix}')")
                with open(file_path, "rb") as f:
                    for i, rec in enumerate(ijson.items(f, prefix)):
                        if i >= max_samples:
                            break
                        print("  샘플:", rec)
            except ImportError:
                print("  [WARNING] ijson이 없어 대용량 파일을 통째로 로드합니다(메모리 주의). "
                      "CELL 3에서 `%pip install -q ijson` 후 재시도를 권장합니다.")
                with open(file_path, encoding="utf-8-sig") as f:
                    obj = json.load(f)
                _peek_loaded_json(obj, max_samples)

        else:
            with open(file_path, encoding="utf-8-sig") as f:
                obj = json.load(f)
            _peek_loaded_json(obj, max_samples)

    except Exception as e:
        print(f"  [ERROR] 파싱 실패: {type(e).__name__}: {e}")


def _peek_loaded_json(obj, max_samples):
    if isinstance(obj, list):
        print(f"  형식: JSON list, 길이: {len(obj)}")
        for rec in obj[:max_samples]:
            print("  샘플:", rec)
    elif isinstance(obj, dict):
        print(f"  형식: JSON dict, 최상위 key: {list(obj.keys())}")
        for key, value in obj.items():
            if isinstance(value, list):
                print(f"    obj['{key}']는 list(길이 {len(value)}) — 레코드 리스트일 가능성 높음, root_key='{key}'")
                for rec in value[:max_samples]:
                    print("    샘플:", rec)


for folder, files in by_folder.items():
    print("=" * 90)
    print(f"[폴더] {folder}  (파일 {len(files)}개)")
    peek_json_structure(files[0])


---

# CELL 12. 공통 텍스트 정제 함수

숫자·의료 단위(`38.5℃`, `10mg`, `120/80`, `3일`, `HbA1c`, `CT`, `MRI`, `COVID-19` 등)·영어
의학 용어는 절대 건드리지 않는다. 처리하는 건: 공백 정규화, zero-width/제어문자 제거,
Unicode 정규화(NFC), 전화번호/이메일 마스킹, 반복문자 판별, 길이 판별뿐이다.

In [ ]:
_ZERO_WIDTH_RE = re.compile(r"[\u200b\u200c\u200d\ufeff]")
_CONTROL_CHAR_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_WS_RE = re.compile(r"\s+")
_PHONE_RE = re.compile(r"(01[016789]|02|0[3-6][1-4])[-. ]?\d{3,4}[-. ]?\d{4}")
_EMAIL_RE = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
_REPEAT_CHAR_RE = re.compile(r"(.)\1{9,}")


def normalize_ws(text):
    if text is None:
        return ""
    return _WS_RE.sub(" ", str(text)).strip()


def clean_text(text):
    """zero-width/제어문자 제거 + Unicode 정규화 + 공백 정규화. 숫자/특수문자/영어 용어는 보존."""
    if text is None:
        return ""
    text = unicodedata.normalize("NFC", str(text))
    text = _ZERO_WIDTH_RE.sub("", text)
    text = _CONTROL_CHAR_RE.sub("", text)
    return normalize_ws(text)


def mask_pii(text):
    if not text or not ENABLE_PII_MASKING:
        return text
    text = _EMAIL_RE.sub("[이메일]", text)
    text = _PHONE_RE.sub("[전화번호]", text)
    return text


def classify_quality_issue(user_text, assistant_text):
    """(문제 있음 여부, 사유)를 반환. 사유는 CELL 21 통계에서 그대로 집계에 쓰인다."""
    if not user_text or not assistant_text:
        return True, "empty"
    if user_text == assistant_text:
        return True, "invalid_answer"
    if len(user_text) < MIN_USER_LENGTH or len(assistant_text) < MIN_ASSISTANT_LENGTH:
        return True, "too_short"
    if len(user_text) > MAX_USER_LENGTH or len(assistant_text) > MAX_ASSISTANT_LENGTH:
        return True, "too_long"
    stripped = _REPEAT_CHAR_RE.sub("", assistant_text)
    if len(stripped) < len(assistant_text) * 0.3:
        return True, "low_quality"
    return False, None


print("[INFO] 공통 정제 함수 정의 완료: normalize_ws / clean_text / mask_pii / classify_quality_issue")


---

# CELL 13. Reasoning 제거 함수

`<think>`/`<analysis>` 블록과 `Reasoning:`/`Chain of Thought:`/`Step N:` 같은 줄 단위
프리픽스를 지운다. 태그/프리픽스만 지우고, 그 뒤에 이어지는 실제 설명 문장은 남긴다.

In [ ]:
_REASONING_BLOCK_RE = re.compile(
    r"<think>.*?</think>|<analysis>.*?</analysis>", re.DOTALL | re.IGNORECASE
)
_REASONING_LINE_PREFIX_RE = re.compile(
    r"^(reasoning|chain of thought|step\s*\d+)\s*[:：]\s*", re.IGNORECASE | re.MULTILINE
)


def strip_reasoning_tags(text):
    if not text:
        return text
    text = _REASONING_BLOCK_RE.sub("", text)
    text = _REASONING_LINE_PREFIX_RE.sub("", text)
    return normalize_ws(text)


_demo = "<think>감별진단을 위해 CBC와 CRP를 우선 고려한다</think>\nStep 1: 발열의 기간을 확인한다\n이 경우 감염 가능성을 우선 고려해야 합니다."
print("[INFO] strip_reasoning_tags 데모")
print("  원본  :", _demo)
print("  제거 후:", strip_reasoning_tags(_demo))


---

# CELL 14. QAAdapter

`question/answer`, `instruction(+input)/output`, `prompt/response` 등 단일 질문-답변 쌍
구조를 공통 `messages`로 바꾼다. `HF_COLUMN_OVERRIDE`에 명시된 컬럼이 있으면 그걸 쓰고,
없으면 흔한 컬럼명 조합을 순서대로 시도한다(자동판별). 둘 다 실패하면 `None`을 반환하고
호출부(CELL 19)에서 `invalid_dialogue` 사유로 집계한다.

In [ ]:
_QA_COLUMN_CANDIDATES = [
    ("question", None, "answer"),
    ("instruction", "input", "output"),
    ("prompt", None, "response"),
]


def QAAdapter(example, dataset_name):
    """단일 질문-답변 쌍 -> messages. 실패하면 None."""
    override = HF_COLUMN_OVERRIDE.get(dataset_name)
    if override:
        q_col, in_col, a_col = override["question_col"], override.get("input_col"), override["answer_col"]
        candidates = [(q_col, in_col, a_col)]
    else:
        candidates = _QA_COLUMN_CANDIDATES

    for q_col, in_col, a_col in candidates:
        if q_col in example and a_col in example:
            question = clean_text(example.get(q_col))
            extra = clean_text(example.get(in_col)) if in_col and example.get(in_col) not in (None, "", "<noinput>") else ""
            answer = clean_text(example.get(a_col))
            if extra:
                question = f"{question}\n\n{extra}"
            if not question or not answer:
                continue
            question, answer = mask_pii(question), mask_pii(answer)
            return [{"role": "user", "content": question}, {"role": "assistant", "content": answer}]

    return None


print("[INFO] QAAdapter 정의 완료")


---

# CELL 15. MCQAdapter

`question` + `A/B/C/D` 선택지를 하나의 user 메시지로 구성한다. 정답은 "B"처럼 글자만
출력하지 않고, **실제 선택지 텍스트를 포함한 완전한 문장**으로 만든다.

`MCQLetteredAdapter`도 같이 정의한다 — `snuh/ClinicalQA`처럼 선택지가 `A`/`B`/`C`/`D`
컬럼이 아니라 `options`라는 **딕셔너리 하나**(`{"option_A": ..., "option_B": ...}`)에 들어있고,
정답 컬럼도 `"C"`처럼 글자 하나뿐인 구조를 위한 것 — 그대로 쓰면 답변이 한 글자라 품질
필터(`MIN_ASSISTANT_LENGTH`)에 걸려 전부 삭제되므로, 선택지 텍스트 + 해설(`explanation`,
있으면)까지 묶어서 완전한 문장으로 만든다.

In [ ]:
def MCQAdapter(example, dataset_name):
    question = clean_text(example.get("question"))
    options_text = "\n".join(
        f"{key}. {clean_text(example[key])}"
        for key in MCQ_OPTION_KEYS
        if example.get(key) is not None and clean_text(example[key])
    )
    if not question or not options_text:
        return None

    answer_letter = None
    try:
        idx = example.get(MCQ_ANSWER_IDX_COL)
        if idx is not None:
            answer_letter = MCQ_OPTION_KEYS[int(idx)]
    except (TypeError, ValueError, IndexError):
        answer_letter = None

    answer_value = example.get(MCQ_ANSWER_TEXT_COL)
    if answer_letter is None and isinstance(answer_value, str) and answer_value.strip().upper() in MCQ_OPTION_KEYS:
        answer_letter = answer_value.strip().upper()

    answer_text = clean_text(example.get(answer_letter)) if answer_letter and example.get(answer_letter) else clean_text(answer_value)
    if not answer_text:
        return None

    user_turn = f"다음 의학 문제에 답하세요.\n\n질문:\n{question}\n\n{options_text}"
    assistant_turn = f"정답은 {answer_letter}입니다.\n\n{answer_letter}. {answer_text}" if answer_letter else f"정답: {answer_text}"

    return [{"role": "user", "content": mask_pii(user_turn)}, {"role": "assistant", "content": mask_pii(assistant_turn)}]


def MCQLetteredAdapter(example, dataset_name):
    """options가 {"option_A": ..., "option_B": ...} 형태의 딕셔너리이고, 정답 컬럼이
    "C"처럼 글자 하나뿐인 데이터용(snuh/ClinicalQA 등)."""
    cfg = HF_COLUMN_OVERRIDE.get(dataset_name, {})
    question = clean_text(example.get(cfg.get("question_col", "question")))
    options = example.get(cfg.get("options_col", "options"))
    if not question or not isinstance(options, dict):
        return None

    options_text = "\n".join(
        f"{key.replace('option_', '').upper()}. {clean_text(val)}"
        for key, val in sorted(options.items())
        if val is not None and clean_text(val)
    )
    if not options_text:
        return None

    answer_letter = str(example.get(cfg.get("answer_col", "answer"), "")).strip().upper()
    answer_text = clean_text(options.get(f"option_{answer_letter}", ""))
    if not answer_letter or not answer_text:
        return None

    explanation_col = cfg.get("explanation_col")
    explanation = clean_text(example.get(explanation_col)) if explanation_col else ""

    user_turn = f"다음 의학 문제에 답하세요.\n\n질문:\n{question}\n\n선택지:\n{options_text}"
    assistant_turn = f"정답은 {answer_letter}입니다.\n\n{answer_letter}. {answer_text}"
    if explanation:
        assistant_turn += f"\n\n해설: {explanation}"

    return [{"role": "user", "content": mask_pii(user_turn)}, {"role": "assistant", "content": mask_pii(assistant_turn)}]


print("[INFO] MCQAdapter / MCQLetteredAdapter 정의 완료")


---

# CELL 16. DialogueAdapter

이미 `messages`/`conversations` 구조인 데이터를 다룬다. `role`이 `human`/`gpt`,
`from`/`value` 등 제각각이어도 `user`/`assistant`로 표준화하고, 멀티턴을 그대로 유지한다.
**반드시 user로 시작해서 assistant로 끝나는 경우만** 유효한 것으로 채택한다.

In [ ]:
def _extract_turns(example):
    for key in ("conversations", "messages", "CoD_conversations", "dialogue", "conversation"):
        value = example.get(key)
        if value:
            return value
    return None


def _standardize_role(turn):
    raw = str(turn.get("from") or turn.get("role") or "").lower()
    if raw in ("doctor", "assistant", "gpt", "bot"):
        return "assistant"
    return "user"


def DialogueAdapter(example, dataset_name):
    turns = _extract_turns(example)
    if not turns or len(turns) < 2:
        return None

    messages = []
    for turn in turns:
        role = _standardize_role(turn)
        content = clean_text(turn.get("value") or turn.get("content") or "")
        if not content:
            continue
        content = mask_pii(content)
        messages.append({"role": role, "content": content})

    if len(messages) < 2 or messages[0]["role"] != "user" or messages[-1]["role"] != "assistant":
        return None

    return messages


print("[INFO] DialogueAdapter 정의 완료")


---

# CELL 17. ReasoningAdapter

QAAdapter/DialogueAdapter가 만든 `messages`를 받아서, assistant 턴에만 CELL 13의
`strip_reasoning_tags()`를 적용하는 후처리 래퍼다. `ENABLE_REASONING_STRIP=False`면 그냥
통과시킨다.

In [ ]:
def ReasoningAdapter(messages):
    if not ENABLE_REASONING_STRIP or not messages:
        return messages
    result = []
    for m in messages:
        if m["role"] == "assistant":
            result.append({"role": "assistant", "content": strip_reasoning_tags(m["content"])})
        else:
            result.append(m)
    return result


print("[INFO] ReasoningAdapter 정의 완료")


---

# CELL 18. AIHubAdapter

`AIHUB_COLUMN_MAPPING`에 지정된 `root_key`(레코드 리스트까지의 경로)와
`question_key`/`answer_key`(단일 QA) 또는 `dialogue_key`(멀티턴)를 이용해 변환한다.
매핑이 비어있거나 실제 레코드에서 해당 key를 못 찾으면, 예외를 던지는 대신 `None`을
반환하고 CELL 20에서 몇 건이 이 사유로 빠졌는지 집계한다.

In [ ]:
def _get_by_path(obj, path):
    if path is None:
        return obj
    for part in path.split("."):
        if isinstance(obj, dict) and part in obj:
            obj = obj[part]
        else:
            return None
    return obj


def AIHubAdapter(record, mapping):
    if mapping.get("question_key") and mapping.get("answer_key"):
        question = clean_text(_get_by_path(record, mapping["question_key"]))
        answer = clean_text(_get_by_path(record, mapping["answer_key"]))
        if not question or not answer:
            return None
        return [{"role": "user", "content": mask_pii(question)}, {"role": "assistant", "content": mask_pii(answer)}]

    if mapping.get("dialogue_key"):
        turns = _get_by_path(record, mapping["dialogue_key"])
        if not turns or not isinstance(turns, list):
            return None
        messages = []
        for turn in turns:
            if not isinstance(turn, dict):
                continue
            role = _standardize_role(turn)
            content = clean_text(turn.get("value") or turn.get("content") or turn.get("text") or "")
            if content:
                messages.append({"role": role, "content": mask_pii(content)})
        if len(messages) < 2 or messages[0]["role"] != "user" or messages[-1]["role"] != "assistant":
            return None
        return messages

    return None  # question_key/answer_key/dialogue_key가 전부 비어있음 — CONFIG 미설정 상태


print("[INFO] AIHubAdapter 정의 완료")


---

# CELL 19. HF 데이터셋 공통 messages 변환

`format_type`에 따라 QAAdapter/MCQAdapter/DialogueAdapter로 분기하고, reasoning 데이터는
ReasoningAdapter까지 통과시킨 뒤 품질 기준(`classify_quality_issue`)으로 최종 채택 여부를
정한다. 제거 사유별 개수를 `conversion_log`에 데이터셋별로 남겨서 CELL 21~22에서 그대로
쓴다.

In [ ]:
converted_rows = []
conversion_log = {}

for d in HF_DATASETS:
    name = d["name"]
    if name not in raw_hf_datasets:
        continue
    ds = raw_hf_datasets[name]
    reason_counts = Counter()
    kept = 0

    for example in ds:
        if d["format_type"] == "qa_pair":
            messages = QAAdapter(example, name)
        elif d["format_type"] == "mcq_abcd":
            messages = MCQAdapter(example, name)
        elif d["format_type"] == "mcq_lettered":
            messages = MCQLetteredAdapter(example, name)
        elif d["format_type"] == "dialogue":
            messages = DialogueAdapter(example, name)
        else:
            messages = None

        if messages is None:
            reason_counts["invalid_dialogue"] += 1
            continue

        if name in REASONING_SOURCE_DATASETS:
            messages = ReasoningAdapter(messages)

        bad, reason = classify_quality_issue(messages[0]["content"], messages[-1]["content"])
        if bad:
            reason_counts[reason] += 1
            continue

        converted_rows.append({
            "messages": messages, "source_dataset": name,
            "data_type": d["data_type"], "format_type": d["format_type"],
        })
        kept += 1

    conversion_log[name] = {"total": len(ds), "kept": kept, "reasons": dict(reason_counts)}
    print(f"[INFO] {name:<45} {len(ds):>6} -> {kept:>6}   {dict(reason_counts)}")

print(f"\n[INFO] HF 데이터셋 변환 완료: {len(converted_rows)}개 (누적)")


---

# CELL 20. AI Hub 데이터 공통 messages 변환

`AIHUB_DATASETS[].path`가 아직 `None`이거나(CELL 6이 못 찾음), `AIHUB_COLUMN_MAPPING`이
placeholder 상태면 건너뛰고 `FAILED_DATASETS`에 "CELL 5의 어떤 값을 채워야 하는지" 사유를
남긴다. `path`는 리스트일 수 있다(원천/라벨링 데이터가 별도 압축인 경우) — 그 안의 모든
폴더를 순회한다. `LARGE_JSON_THRESHOLD_MB` 이상인 `.json` 파일은 `ijson`으로 스트리밍
읽어서 한 번에 메모리에 안 올린다.

In [ ]:
def iter_json_records(file_path, root_key=None):
    """대용량 JSON은 ijson 스트리밍으로, 작은 파일/jsonl/csv는 일반적인 방식으로 레코드를 하나씩 낸다."""
    if file_path.endswith(".jsonl"):
        with open(file_path, encoding="utf-8-sig") as f:
            for line in f:
                line = line.strip()
                if line:
                    yield json.loads(line)
        return

    if file_path.endswith(".csv"):
        yield from pd.read_csv(file_path).to_dict("records")
        return

    if file_path.endswith(".parquet"):
        yield from pd.read_parquet(file_path).to_dict("records")
        return

    size_mb = os.path.getsize(file_path) / 1024 / 1024
    if size_mb >= LARGE_JSON_THRESHOLD_MB:
        try:
            import ijson
            prefix = f"{root_key}.item" if root_key else "item"
            with open(file_path, "rb") as f:
                yield from ijson.items(f, prefix)
            return
        except ImportError:
            print(f"  [WARNING] {file_path}: {size_mb:.0f}MB인데 ijson이 없어 통째로 로드합니다(메모리 주의).")

    with open(file_path, encoding="utf-8-sig") as f:
        obj = json.load(f)
    root = _get_by_path(obj, root_key) if isinstance(obj, dict) else obj
    yield from (root if isinstance(root, list) else [obj])


for d in AIHUB_DATASETS:
    name = d["name"]
    mapping = AIHUB_COLUMN_MAPPING.get(name, {})

    if not d["path"]:
        print(f"[WARNING] {name}: path가 아직 연결되지 않았습니다 — CELL 6의 archive_names 매칭 결과를 "
              f"확인하거나 CELL 5에서 직접 채우세요. 건너뜁니다.")
        FAILED_DATASETS.append({"name": name, "stage": "path_not_set", "reason": "AIHUB_DATASETS[].path가 None"})
        continue

    if not (mapping.get("question_key") or mapping.get("dialogue_key")):
        print(f"[WARNING] {name}: AIHUB_COLUMN_MAPPING이 아직 비어있습니다 — "
              f"CELL 11 결과를 보고 CELL 5의 AIHUB_COLUMN_MAPPING['{name}']을 채우세요. 건너뜁니다.")
        FAILED_DATASETS.append({"name": name, "stage": "mapping_not_set", "reason": "AIHUB_COLUMN_MAPPING 비어있음"})
        continue

    paths = d["path"] if isinstance(d["path"], list) else [d["path"]]
    found_files = []
    for folder_path in paths:
        if not os.path.isdir(folder_path):
            print(f"[ERROR] {name}: 폴더가 존재하지 않습니다: {folder_path}")
            continue
        for ext in ("*.json", "*.jsonl", "*.csv", "*.parquet"):
            found_files.extend(glob.glob(os.path.join(folder_path, "**", ext), recursive=True))

    reason_counts = Counter()
    kept = 0
    total = 0

    for fp in found_files:
        try:
            records_iter = iter_json_records(fp, mapping.get("root_key"))
        except Exception as e:
            print(f"[ERROR] {fp} 로드 실패: {type(e).__name__}: {e}")
            continue

        for record in records_iter:
            total += 1
            try:
                messages = AIHubAdapter(record, mapping)
            except Exception:
                reason_counts["parse_error"] += 1
                continue
            if messages is None:
                reason_counts["invalid_dialogue"] += 1
                continue
            if ENABLE_REASONING_STRIP:
                messages = ReasoningAdapter(messages)
            bad, reason = classify_quality_issue(messages[0]["content"], messages[-1]["content"])
            if bad:
                reason_counts[reason] += 1
                continue
            converted_rows.append({
                "messages": messages, "source_dataset": name,
                "data_type": d["data_type"], "format_type": d["format_type"],
            })
            kept += 1

    conversion_log[name] = {"total": total, "kept": kept, "reasons": dict(reason_counts)}
    print(f"[INFO] {name:<45} {total:>6} -> {kept:>6}   {dict(reason_counts)}")

print(f"\n[INFO] 전체 변환 완료 (HF + AI Hub 누적): {len(converted_rows)}개")
print(f"[INFO] FAILED_DATASETS: {len(FAILED_DATASETS)}건")
for f in FAILED_DATASETS:
    print(f"  - {f['name']} ({f['stage']}): {f['reason']}")


---

# CELL 21. 데이터 품질 검증

`conversion_log`(CELL 19~20에서 데이터셋별로 쌓은 제거 사유)를 전체 합산해서 원본/최종
개수, 사유별 제거 개수를 보여주고, 실제 남은 데이터의 길이 통계와 랜덤 샘플 10개를
출력한다.

In [ ]:
total_before = sum(v["total"] for v in conversion_log.values())
total_kept = sum(v["kept"] for v in conversion_log.values())

reason_totals = Counter()
for v in conversion_log.values():
    reason_totals.update(v["reasons"])

print("=" * 60)
print("데이터 품질 검증 결과 (전체 합산)")
print("=" * 60)
print(f"원본 총합     : {total_before}")
print(f"최종 남은 개수 : {total_kept} ({total_kept/total_before:.1%})" if total_before else "원본 없음")
print("\n제거 사유별 개수:")
for reason, n in reason_totals.most_common():
    print(f"  {reason:<20} {n}")

turn_counts = [len(r["messages"]) for r in converted_rows]
user_lens = [len(r["messages"][0]["content"]) for r in converted_rows]
assistant_lens = [len(r["messages"][-1]["content"]) for r in converted_rows]

print(f"\n평균 대화 턴 수  : {sum(turn_counts)/len(turn_counts):.2f}")
print(f"평균 user 길이   : {sum(user_lens)/len(user_lens):.1f}  (min={min(user_lens)}, max={max(user_lens)})")
print(f"평균 assistant 길이: {sum(assistant_lens)/len(assistant_lens):.1f}  (min={min(assistant_lens)}, max={max(assistant_lens)})")

print("\n랜덤 샘플 10개:")
random.seed(SEED)
for row in random.sample(converted_rows, min(10, len(converted_rows))):
    print("-" * 60)
    print(f"[{row['source_dataset']} / {row['data_type']} / {row['format_type']}]")
    print("user     :", row["messages"][0]["content"][:150])
    print("assistant:", row["messages"][-1]["content"][:150])


---

# CELL 22. Dataset별 전처리 통계 출력

CELL 21이 "전체 합산"이었다면, 이 셀은 **데이터셋별로 나눠서** 표로 정리한 버전이다.

In [ ]:
stats_rows = []
for name, log in conversion_log.items():
    src = next((d for d in HF_DATASETS + AIHUB_DATASETS if d["name"] == name), {})
    stats_rows.append({
        "source_dataset": name,
        "data_type": src.get("data_type"),
        "format_type": src.get("format_type"),
        "total": log["total"],
        "kept": log["kept"],
        "kept_ratio": f"{log['kept']/log['total']:.1%}" if log["total"] else "-",
    })

stats_df = pd.DataFrame(stats_rows).sort_values("data_type").reset_index(drop=True)
print(stats_df.to_string(index=False))

print(f"\n[INFO] 처리 성공 데이터셋: {len(conversion_log)}/{len(HF_DATASETS) + len(AIHUB_DATASETS)}개 "
      f"(실패: {len(FAILED_DATASETS)}개)")


---

# CELL 23. 14개 데이터셋 병합

지금까지 쌓인 `converted_rows`(HF 11개 + AI Hub 3개 중 처리 성공분)를 하나의 `Dataset`으로
합친다.

In [ ]:
merged_dataset = Dataset.from_list(converted_rows)
print(f"[INFO] 병합 완료: 총 {len(merged_dataset)}개")
print(f"[INFO] 참여 데이터셋 수: {len(set(merged_dataset['source_dataset']))}개 "
      f"(목표 14개 중 {len(conversion_log)}개 성공)")


---

# CELL 24. Cross-Dataset 중복 제거

병합된 전체 데이터셋 기준으로 `messages` 내용이 완전히 동일한 행을 제거한다. 동일한 내용이
여러 데이터셋에 있으면 `DATASET_PRIORITY`(CELL 5)상 우선순위가 높은 쪽의 `source_dataset`을
남긴다 — AI Hub 공신력 데이터 > ClinicalQA > Asan/KoMedInstruct > 나머지 knowledge >
reasoning > conversation_style 순.

In [ ]:
def _priority_rank(source_dataset):
    try:
        return DATASET_PRIORITY.index(source_dataset)
    except ValueError:
        return len(DATASET_PRIORITY)  # 목록에 없으면 최하위 우선순위


if not ENABLE_CROSS_DATASET_DUPLICATE_REMOVAL:
    print("[INFO] ENABLE_CROSS_DATASET_DUPLICATE_REMOVAL=False — 건너뜀")
else:
    groups = defaultdict(list)
    for i, row in enumerate(merged_dataset):
        key = tuple((m["role"], m["content"]) for m in row["messages"])
        groups[key].append(i)

    keep_indices = []
    removed = 0
    for key, idxs in groups.items():
        if len(idxs) == 1:
            keep_indices.append(idxs[0])
            continue
        best_idx = min(idxs, key=lambda i: _priority_rank(merged_dataset[i]["source_dataset"]))
        keep_indices.append(best_idx)
        removed += len(idxs) - 1

    before_n = len(merged_dataset)
    merged_dataset = merged_dataset.select(sorted(keep_indices))
    print(f"[INFO] Cross-Dataset 중복 제거: {before_n} -> {len(merged_dataset)}개 ({removed}개 제거)")


---

# CELL 25. data_type 비율 확인 및 Dataset Mixing

`data_type`/`source_dataset` 분포를 있는 그대로 보여준다. `ENABLE_MIXING=True`일 때만
`TARGET_RATIOS`에 맞춰 **다운샘플링만** 수행한다(복제로 오버샘플링하지 않음). 가장 적은
그룹이 병목이 되므로, 그 그룹 기준으로 달성 가능한 총량을 먼저 계산하고 목표치를 못 채우는
그룹은 경고만 출력한다.

In [ ]:
type_counts = Counter(merged_dataset["data_type"])
total_n = len(merged_dataset)

print("data_type별 분포:")
for dtype, n in type_counts.items():
    print(f"  {dtype:<20} {n:>6}개 ({n/total_n:.1%})")

print("\nsource_dataset별 분포:")
for src, n in Counter(merged_dataset["source_dataset"]).most_common():
    print(f"  {src:<45} {n:>6}개")

if not ENABLE_MIXING:
    print("\n[INFO] ENABLE_MIXING=False — 원본 비율을 그대로 사용합니다.")
else:
    feasible_total = min(
        type_counts[dtype] / ratio
        for dtype, ratio in TARGET_RATIOS.items()
        if type_counts.get(dtype, 0) > 0 and ratio > 0
    )
    by_type_indices = defaultdict(list)
    for i, dtype in enumerate(merged_dataset["data_type"]):
        by_type_indices[dtype].append(i)

    rng = random.Random(SEED)
    keep_indices = []
    for dtype, ratio in TARGET_RATIOS.items():
        available = by_type_indices.get(dtype, [])
        target_n = int(feasible_total * ratio)
        if target_n > len(available):
            print(f"[WARNING] {dtype}: 목표 {target_n}개를 채울 수 없습니다(보유 {len(available)}개) — 있는 만큼만 사용")
            target_n = len(available)
        keep_indices.extend(rng.sample(available, target_n) if target_n < len(available) else available)

    merged_dataset = merged_dataset.select(sorted(keep_indices))
    print(f"\n[INFO] Mixing 적용 후: {len(merged_dataset)}개")
    for dtype, n in Counter(merged_dataset["data_type"]).items():
        print(f"  {dtype:<20} {n:>6}개 ({n/len(merged_dataset):.1%})")


---

# CELL 26. Train / Validation / Test Split

분리 전에 정규화된 첫 user 질문 기준으로 한 번 더 중복을 제거해서(다른 데이터셋 간에 같은
질문이 실려 있는 경우 Train/Test에 걸쳐 들어가는 걸 방지) 데이터 누수를 줄인다. 이후
`data_type` 그룹별로 각각 80/10/10으로 나눈 뒤 합치는 방식(Stratified Split)으로, 작은
그룹(conversation_style 등)이 특정 split에 몰리지 않게 한다.

In [ ]:
seen_q = set()
keep_idx = []
for i, row in enumerate(merged_dataset):
    q_key = normalize_ws(row["messages"][0]["content"]).lower()
    if q_key in seen_q:
        continue
    seen_q.add(q_key)
    keep_idx.append(i)

before_n = len(merged_dataset)
merged_dataset = merged_dataset.select(keep_idx)
print(f"[INFO] 질문 기준 추가 중복 제거: {before_n} -> {len(merged_dataset)}개")

train_parts, val_parts, test_parts = [], [], []
for dtype in sorted(set(merged_dataset["data_type"])):
    subset = merged_dataset.filter(lambda r, dt=dtype: r["data_type"] == dt)
    split1 = subset.train_test_split(test_size=0.2, seed=SEED)
    split2 = split1["test"].train_test_split(test_size=0.5, seed=SEED)
    train_parts.append(split1["train"])
    val_parts.append(split2["train"])
    test_parts.append(split2["test"])
    print(f"[INFO] {dtype:<20} train={len(split1['train']):>5} val={len(split2['train']):>5} test={len(split2['test']):>5}")

train_dataset = concatenate_datasets(train_parts).shuffle(seed=SEED)
val_dataset = concatenate_datasets(val_parts).shuffle(seed=SEED)
test_dataset = concatenate_datasets(test_parts).shuffle(seed=SEED)

print(f"\n[INFO] Train: {len(train_dataset)}개 ({len(train_dataset)/len(merged_dataset):.1%})")
print(f"[INFO] Val  : {len(val_dataset)}개 ({len(val_dataset)/len(merged_dataset):.1%})")
print(f"[INFO] Test : {len(test_dataset)}개 ({len(test_dataset)/len(merged_dataset):.1%})")


---

# CELL 27. 최종 Dataset 검증

각 split이 실제로 "user로 시작해서 assistant로 끝나고, 빈 content가 없는" 유효한 구조인지
마지막으로 한 번 더 확인한다. `data_type` 비율과 `source_dataset` 분포도 split별로 출력한다.

In [ ]:
def validate_split(name, ds):
    issues = 0
    for row in ds:
        msgs = row["messages"]
        if not msgs or msgs[0]["role"] != "user" or msgs[-1]["role"] != "assistant":
            issues += 1
            continue
        if any(not m["content"] for m in msgs):
            issues += 1
    print(f"[INFO] {name}: {len(ds)}개, 구조 이상 {issues}건")
    print(f"  data_type 비율   : {dict(Counter(ds['data_type']))}")
    print(f"  source_dataset top5: {Counter(ds['source_dataset']).most_common(5)}")
    return issues


total_issues = 0
for split_name, ds in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    total_issues += validate_split(split_name, ds)

if total_issues:
    print(f"\n[WARNING] 구조 이상 {total_issues}건 — 저장 전에 원인을 확인하세요.")
else:
    print("\n[INFO] 전체 split 구조 검증 통과")


---

# CELL 28. JSONL 및 Hugging Face Dataset 저장

`final_sft_dataset.jsonl`(전체), `train.jsonl`/`validation.jsonl`/`test.jsonl`,
`dataset_statistics.json`, `preprocessing_log.json`(데이터셋별 제거 사유 원본 로그),
`dataset_config.json`(이번 실행에 쓰인 CONFIG 스냅샷), 그리고 Hugging Face `Dataset` 형식을
`processed_dataset/`에 저장한다. 전부 UTF-8, `ensure_ascii=False`로 저장해서 한글이 깨지지
않게 한다.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)


def _to_jsonl(ds, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in ds:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"[INFO] 저장: {path} ({len(ds)}개)")


_to_jsonl(merged_dataset, os.path.join(OUTPUT_DIR, "final_sft_dataset.jsonl"))
_to_jsonl(train_dataset, os.path.join(OUTPUT_DIR, "train.jsonl"))
_to_jsonl(val_dataset, os.path.join(OUTPUT_DIR, "validation.jsonl"))
_to_jsonl(test_dataset, os.path.join(OUTPUT_DIR, "test.jsonl"))

stats = {
    "total": len(merged_dataset),
    "train": len(train_dataset), "validation": len(val_dataset), "test": len(test_dataset),
    "data_type_counts": dict(Counter(merged_dataset["data_type"])),
    "source_dataset_counts": dict(Counter(merged_dataset["source_dataset"])),
    "failed_datasets": FAILED_DATASETS,
    "failed_archives": FAILED_ARCHIVES,
}
with open(os.path.join(OUTPUT_DIR, "dataset_statistics.json"), "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2, default=str)
print("[INFO] 저장: dataset_statistics.json")

with open(os.path.join(OUTPUT_DIR, "preprocessing_log.json"), "w", encoding="utf-8") as f:
    json.dump({"conversion_log": conversion_log, "archive_extract_log": ARCHIVE_EXTRACT_LOG},
               f, ensure_ascii=False, indent=2, default=str)
print("[INFO] 저장: preprocessing_log.json")

config_snapshot = {
    "SEED": SEED, "HF_DATASETS": HF_DATASETS, "AIHUB_DATASETS": AIHUB_DATASETS,
    "AIHUB_COLUMN_MAPPING": AIHUB_COLUMN_MAPPING, "ARCHIVE_ROOT": ARCHIVE_ROOT, "EXTRACT_ROOT": EXTRACT_ROOT,
    "MIN_USER_LENGTH": MIN_USER_LENGTH, "MIN_ASSISTANT_LENGTH": MIN_ASSISTANT_LENGTH,
    "MAX_USER_LENGTH": MAX_USER_LENGTH, "MAX_ASSISTANT_LENGTH": MAX_ASSISTANT_LENGTH,
    "ENABLE_REASONING_STRIP": ENABLE_REASONING_STRIP,
    "ENABLE_DUPLICATE_REMOVAL": ENABLE_DUPLICATE_REMOVAL,
    "ENABLE_CROSS_DATASET_DUPLICATE_REMOVAL": ENABLE_CROSS_DATASET_DUPLICATE_REMOVAL,
    "ENABLE_PII_MASKING": ENABLE_PII_MASKING, "ENABLE_MIXING": ENABLE_MIXING,
    "TARGET_RATIOS": TARGET_RATIOS, "DATASET_PRIORITY": DATASET_PRIORITY, "OUTPUT_DIR": OUTPUT_DIR,
}
with open(os.path.join(OUTPUT_DIR, "dataset_config.json"), "w", encoding="utf-8") as f:
    json.dump(config_snapshot, f, ensure_ascii=False, indent=2, default=str)
print("[INFO] 저장: dataset_config.json")

hf_dataset_dir = os.path.join(OUTPUT_DIR, "processed_dataset")
merged_dataset.save_to_disk(hf_dataset_dir)
print(f"[INFO] Hugging Face Dataset 저장: {hf_dataset_dir}")


---

# CELL 29. 최종 결과 요약 및 랜덤 샘플 출력

In [ ]:
print("=" * 60)
print("최종 결과 요약")
print("=" * 60)
print(f"목표 데이터셋 : 14개 (HF 11 + AI Hub 3)")
print(f"처리 성공     : {len(conversion_log)}개")
print(f"처리 실패     : {len(FAILED_DATASETS)}개")
print(f"압축 해제 실패: {len(FAILED_ARCHIVES)}개")
print(f"최종 전체     : {len(merged_dataset)}개")
print(f"  Train       : {len(train_dataset)}개")
print(f"  Validation  : {len(val_dataset)}개")
print(f"  Test        : {len(test_dataset)}개")
print(f"저장 위치     : {OUTPUT_DIR}")

if FAILED_DATASETS:
    print("\n실패한 데이터셋:")
    for f in FAILED_DATASETS:
        print(f"  - {f['name']} ({f['stage']}): {f['reason']}")

if FAILED_ARCHIVES:
    print("\n실패한 압축파일:")
    for f in FAILED_ARCHIVES:
        print(f"  - {f['archive']}: {f['reason']}")

print("\n랜덤 샘플 5개 (train):")
random.seed(SEED)
for row in random.sample(list(train_dataset), min(5, len(train_dataset))):
    print("-" * 60)
    print(f"[{row['source_dataset']} / {row['data_type']}]")
    for m in row["messages"]:
        print(f"  {m['role']}: {m['content'][:150]}")

print("\n=== 데이터 파이프라인 완료 — 이 산출물을 학습 노트북에서 불러와 사용하세요 ===")
